### Final model

Goal: To test different hyperparameters for XGB Model

In notebook 03 we analyzed that:
- class_weight = 'balanced' is best imbalance handling method due to speed and simplicity, there is almost no performance differences between it and SMOTE (while Undersampling underperformed)
- XGBoost gives us the best PR-AUC and Recall (compared to Random Forest, LightGBM)

This notebook will:
1. Tune XGBoost hyperparameters with Optuna (50 trials, optimizing PR-AUC)
2. SHAP analysis (global + local explanations)
3. Threshold tuning (find best operating point)
4. Final evaluation on test set

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

from imblearn.pipeline import Pipeline
from xgboost import XGBClassifier

import optuna
import shap

from sklearn.metrics import confusion_matrix, precision_recall_curve, classification_report

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data = pd.read_csv('../data/creditcard.csv')

X = data.drop(columns=["Class"])
y = data["Class"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), ['Amount', "Time"])
    ],
    remainder='passthrough'
)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

### Hyperparameter tuning for XGBoost with Optuna

50 trials. We tune 7 parameters:
- n_estimators, max_depth, learning_rate (tree structure)
- subsample, colsample_bytree (regularization through randomness)
- reg_alpha, reg_lambda (L1/L2 regularization)

Objective: maximize PR-AUC on 5-fold StratifiedKFold CV

In [ ]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }

    pipeline_XGB = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(**params, scale_pos_weight=scale_pos_weight, random_state=42))
    ])

    mean_pr_auc = cross_validate(pipeline_XGB, X_train, y_train, cv=skf, scoring=['average_precision'], n_jobs=4)['test_average_precision'].mean()

    return mean_pr_auc

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, n_jobs=1, show_progress_bar=True)

In [ ]:
print("Best PR-AUC:", study.best_value)
print("Best params:", study.best_trial.params)

### Final model training

Train XGBoost with best hyperparameters from Optuna on the full training set, then predict probabilities on the test set.

In [ ]:
# Best params from Optuna run. To rerun tuning, run study.optimize() above
best_params = {
    'n_estimators': 281,
    'max_depth': 5,
    'learning_rate': 0.14963094122519882,
    'subsample': 0.5959511317621005,
    'colsample_bytree': 0.5574131751181338,
    'reg_alpha': 0.04279473702807638,
    'reg_lambda': 9.195603779116694e-05,
}

print("Using best params from previous Optuna run:")
print(f"PR-AUC: 0.8606")
print(best_params)

In [ ]:
pipeline_final = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(**best_params, scale_pos_weight=scale_pos_weight, random_state=42))
    ])

pipeline_final.fit(X_train, y_train)

y_pred_proba = pipeline_final.predict_proba(X_test)[:, 1]

print("Shape:", y_pred_proba.shape)
print("Min:", y_pred_proba.min())
print("Max:", y_pred_proba.max())
print("Mean:", y_pred_proba.mean())

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)

### Threshold tuning

We test three thresholds:
1. **Default (0.5)** - baseline reference
2. **Best F1** - mathematical optimum
3. **Max recall with precision >= 0.5** - business-driven (catch as many frauds as possible)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision_for_thresholds = precision[:-1]
recall_for_thresholds = recall[:-1]

# F1 scores dla wszystkich progów (będzie potrzebne 2 razy)
f1_scores = 2 * precision_for_thresholds * recall_for_thresholds / (
    precision_for_thresholds + recall_for_thresholds + 1e-10
)

# === 1. Default threshold (0.5) ===
y_pred_default = (y_pred_proba >= 0.5).astype(int)

print("=== Default threshold (0.5) ===")
print(f"Threshold: 0.5000")
print(f"Precision: {precision_score(y_test, y_pred_default):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred_default):.3f}")
print(f"F1:        {f1_score(y_test, y_pred_default):.3f}")
print()

# === 2. Best F1 threshold ===
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("=== Best F1 threshold ===")
print(f"Threshold: {thresholds[best_idx]:.4f}")
print(f"Precision: {precision_for_thresholds[best_idx]:.3f}")
print(f"Recall:    {recall_for_thresholds[best_idx]:.3f}")
print(f"F1:        {f1_scores[best_idx]:.3f}")
print()

# === 3. Max recall with precision >= 0.5 ===
mask = precision_for_thresholds >= 0.5
recall_constrained = np.where(mask, recall_for_thresholds, 0)
best_recall_idx = np.argmax(recall_constrained)

print("=== Max recall (precision >= 0.5) ===")
print(f"Threshold: {thresholds[best_recall_idx]:.4f}")
print(f"Precision: {precision_for_thresholds[best_recall_idx]:.3f}")
print(f"Recall:    {recall_for_thresholds[best_recall_idx]:.3f}")
print(f"F1:        {f1_scores[best_recall_idx]:.3f}")

### Threshold choice

The three thresholds show clear trade-offs:
- **Best F1 (0.96)** gives highest precision (96%) but slightly lower recall (82%)
- **Default (0.5)** has best balance of recall (85%) and precision (88%)
- **Max recall (0.007)** catches 90% of frauds but generates many false alarms (50% precision)

**Selected: Best F1 threshold (0.96)** for the final model. Reasoning:
- High precision means fewer false alarms (less operational cost)
- Recall stays high (82%, catches 4 out of 5 frauds)
- F1 = 0.884 (best balance)
- Business can adjust threshold later based on actual cost matrix

In [ ]:
y_pred = (y_pred_proba >= best_threshold).astype(int)
y_pred

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm

### Confusion matrix interpretation

Total frauds in test: 98 (80 + 18)
- Caught: 80 (82%)
- Missed: 18 (18%)

Total alarms: 83 (80 + 3)
- True: 80 (96%)
- False: 3 (4%)

In [ ]:
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legit', 'Fraud'],
            yticklabels=['Legit', 'Fraud'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (threshold 0.96)')
plt.show()

In [ ]:
final_model = pipeline_final.named_steps['classifier']
preprocessor = pipeline_final.named_steps['preprocessor']

X_test_transformed = preprocessor.transform(X_test)

explainer = shap.Explainer(final_model)
shap_values = explainer(X_test_transformed)

In [ ]:
print("Shape:", shap_values.values.shape)

In [ ]:
feature_names = ['Amount', 'Time'] + [f'V{i}' for i in range(1, 29)]
shap_values.feature_names = feature_names

In [ ]:
shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.title("SHAP: Top 15 features")
plt.tight_layout()
plt.show()

In [ ]:
neutral_threshold = 0.9581

idx_fraud = np.argmax(y_pred_proba)
idx_legit = np.argmin(y_pred_proba)
idx_neutral = np.argmin(np.abs(y_pred_proba - neutral_threshold))

print(f"High-risk transaction (idx={idx_fraud}): proba={y_pred_proba[idx_fraud]:.4f}, actual={y_test.iloc[idx_fraud]}")
print(f"Borderline transaction (idx={idx_neutral}): proba={y_pred_proba[idx_neutral]:.4f}, actual={y_test.iloc[idx_neutral]}")
print(f"Legit transaction (idx={idx_legit}): proba={y_pred_proba[idx_legit]:.4f}, actual={y_test.iloc[idx_legit]}")

In [ ]:
print("Shape:", shap_values.values.shape)

In [ ]:
shap.plots.waterfall(shap_values[idx_fraud], show=False)
plt.title("High-risk transaction (predicted fraud)")
plt.show()

### High-risk transaction (idx=29865)

Model predicts fraud with probability 1.0. Actual label = fraud (correct)

Top drivers: low V10, V14, V12 and high V4: classic fraud signals consistent with the global SHAP analysis above.


In [ ]:
shap.plots.waterfall(shap_values[idx_neutral], show=False)
plt.title("Borderline transaction")
plt.show()

### Borderline transaction (idx=12266)

Probability 0.9581, exactly at our decision threshold. Actual label: fraud

V14 and V4 push toward fraud strongly enough to clear the 0.96 threshold, but most other features push toward legit. This transaction is right on the edge of being missed.

In [ ]:
shap.plots.waterfall(shap_values[idx_legit], show=False)
plt.title("Low-risk transaction (predicted legit)")
plt.show()

### Legit transaction (idx=8229)

Probability ~0, zero signs of a fraud. Actual label: legit

All features push toward legit. Model is very confident

### Final summary

This project built a fraud detection system on the Credit Card Fraud Detection dataset (284k transactions, 0.17% fraud rate)

**Approach:**
- EDA: identified imbalance + key features
- Baseline: LR and RF
- Imbalance handling: tested 3 techniques, selected class_weight
- Model selection: XGB > LGBM > RF on PR-AUC
- Tuning: Optuna 50 trials (PR-AUC 0.855 → 0.860)
- Threshold tuning: selected 0.96 for max F1 (precision 0.96, recall 0.82)
- Interpretability: SHAP global + local explanations

**Final model performance (test set, threshold 0.96):**
- Precision: 0.964
- Recall: 0.816
- F1: 0.884
- Caught 80/98 frauds with only 3 false alarms (in 57k test transactions)